# RadFlow Captioning Module Evaluation

This notebook demonstrates how we used OpenAI's Evals framework for captioning task. Leveraging the Evals API, we graded model-generated captions to generated frames and prompt by using **model grading** (LLM as a Judge) to score the model responses against the image.

In this example, we will evaluate how well our model can:
1. **Generate patient-friendly captions** to educate patients gently
3. **Be medically accurate** to prevent misunderstanding

## Installing Dependencies + Setup

In [1]:
# Install required packages
!pip install openai datasets pandas --quiet

In [2]:
# Import libraries
from datasets import load_dataset
from openai import OpenAI
import os
import json
import time
import pandas as pd

## Dataset Preparation

We used 5 examples of our generated data.


In [36]:
prompt_v1 = '''
You are implementing the RadFlow image captioning module.

Input:
- A compact sequential X-ray image showing three stages:
  1. fractured/pre-operative stage
  2. fixation/implant stage
  3. healing stage

Perform the following three phases:

Phase 1:
Generate a medically accurate caption describing the full X-ray sequence, including bone and
implant names.

Phase 2:
Rewrite it into a patient-friendly explanation that:
- speaks directly to the patient using "your"
- explains what the doctors did during surgery in simple terms
- uses a calm, reassuring tone
- avoids scary or harsh wording
- use non-medical terms for bone name

Phase 3:
Align the explanation with each temporal frame:
- Frame 1: fracture stage
- Frame 2: fixation/implant stage
- Frame 3: healing stage

For each frame:
- speak directly to the patient
- explain what is happening clearly
- keep the tone calm and reassuring
- write everything in future tense because this explanation will be shown before surgery
- describe what the patient will see and what doctors will do

Your explanations must be in 2-4 sentences.

Return ONLY valid JSON in this structure:
{{
  "phase_1_medical_caption": "",
  "phase_2_patient_friendly_caption": "",
  "phase_3_alignment": [
    {{"frame": 1, "stage": "fracture", "explanation": ""}},
    {{"frame": 2, "stage": "fixation", "explanation": ""}},
    {{"frame": 3, "stage": "healing", "explanation": ""}}
  ]
}}
'''

In [38]:
dataset = [{"image_url": "https://raw.githubusercontent.com/Ghaaidda/temp/main/1.jpeg", "prompt": prompt_v1},
           {"image_url": "https://raw.githubusercontent.com/Ghaaidda/temp/main/2.jpeg", "prompt": prompt_v1},
           {"image_url": "https://raw.githubusercontent.com/Ghaaidda/temp/main/3.jpg", "prompt": prompt_v1},
           {"image_url": "https://raw.githubusercontent.com/Ghaaidda/temp/main/4.jpg", "prompt": prompt_v1},
           {"image_url": "https://raw.githubusercontent.com/Ghaaidda/temp/main/5.jpg", "prompt": prompt_v1}]

We extract the relevant fields and put it in a json-like format to pass in as a data source in the Evals API. Input image data can be in the form of a web URL or a base64 encoded string. Here, we use the provided web URLs.

In [51]:
evals_data_source = []

for example in dataset:
    evals_data_source.append({
        "item": {
            "image_url": example["image_url"], # image web URL
            "prompt": example["prompt"] # prompt
        }
    })

## Eval Configuration

In [42]:
from google.colab import userdata
client = OpenAI(
    api_key=userdata.get('OPENAI_API_KEY')
)

Evals have two parts, the "Eval" and the "Run". In the "Eval", we define the expected structure of the data and the testing criteria (grader).

### Data Source Config


In [43]:
data_source_config = {
    "type": "custom",
    "item_schema": {
        "type": "object",
        "properties": {
          "image_url": { "type": "string" },
          "prompt": { "type": "string" }
        },
        "required": ["image_url", "prompt"]
      },
    "include_sample_schema" : True
}

### Testing Criteria

Our main criterias for the captioning module is the tone and medical accuracy.

In [44]:
Tone_grader_config = {
	    "type": "label_model",
        "name": "Tone Grader",
        "input":[
            {
                "role": "system",
		        "content": "You are an expert grader. Judge the tone of the model response, if its harsh or too cold and robotic its a Fail, if its reassuring and patient friendly its a Pass"},
	          {
		        "role": "user",
		        "content": [
							{ "type": "input_text", "text": "Prompt: {{ item.prompt }}. Model response: {{ sample.output_text }}." },
							{ "type": "input_image", "image_url": "{{ item.image_url }}", "detail": "auto" },
							]
	        }
		],
		"labels": ["Pass", "Fail"],
		"passing_labels": ["Pass"],
	    "model": "o4-mini" # model for grading
	}

In [45]:
med_acc_grader_config = {
	    "type": "label_model",
        "name": "Medical Accuracy Grader",
        "input":[
            {
                "role": "system",
		        "content": "You are an expert grader. Judge the medical accuracy of the model response, if its medically inacruate or misleading its a Fail, if its medically sound and correct its a Pass."},
	          {
		        "role": "user",
		        "content": [
							{ "type": "input_text", "text": "Prompt: {{ item.prompt }}. Model response: {{ sample.output_text }}." },
							{ "type": "input_image", "image_url": "{{ item.image_url }}", "detail": "auto" },
							]
	        }
		],
		"labels": ["Pass", "Fail"],
		"passing_labels": ["Pass"],
	    "model": "o4-mini" # model for grading
	}

Now, we create the eval object.

In [46]:
eval_object1 = client.evals.create(
        name="Image Grading",
        data_source_config=data_source_config,
        testing_criteria=[Tone_grader_config],
    )

In [47]:
eval_object2 = client.evals.create(
        name="Image Grading",
        data_source_config=data_source_config,
        testing_criteria=[med_acc_grader_config],
    )

## Eval Run

To create the run, we pass in the eval object id, the data source and the chat message input we will use for sampling to generate the model response.

In [48]:
sampling_messages = [{
    "role": "user",
    "type": "message",
    "content": {
        "type": "input_text",
        "text": "{{ item.prompt }}"
      }
  },
  {
    "role": "user",
    "type": "message",
    "content": {
        "type": "input_image",
        "image_url": "{{ item.image_url }}",
        "detail": "auto"
    }
  }]

We now kickoff an eval run.

In [53]:
eval_run1 = client.evals.runs.create(
        name="Image Input Eval Run",
        eval_id=eval_object1.id,
        data_source={
            "type": "responses", # sample using responses API
            "source": {
                "type": "file_content",
                "content": evals_data_source
            },
            "model": "gpt-4.1", # model used to generate the response
            "input_messages": {
                "type": "template",
                "template": sampling_messages}
        }
    )

In [54]:
eval_run2 = client.evals.runs.create(
        name="Image Input Eval Run",
        eval_id=eval_object2.id,
        data_source={
            "type": "responses", # sample using responses API
            "source": {
                "type": "file_content",
                "content": evals_data_source
            },
            "model": "gpt-4.1", # model used to generate the response
            "input_messages": {
                "type": "template",
                "template": sampling_messages}
        }
    )

## Poll and Display the Results

When the run finishes, we can take a look at the result.

In [55]:
while True:
    run1 = client.evals.runs.retrieve(run_id=eval_run1.id, eval_id=eval_object1.id)
    run2 = client.evals.runs.retrieve(run_id=eval_run2.id, eval_id=eval_object2.id)
    if run1.status == "completed" or run1.status == "failed": # check if the run is finished
        output_items1 = list(client.evals.runs.output_items.list(
            run_id=run1.id, eval_id=eval_object1.id
        ))
        output_items2 = list(client.evals.runs.output_items.list(
            run_id=run2.id, eval_id=eval_object2.id
        ))
        df = pd.DataFrame({
                "prompt": [item.datasource_item["prompt"] for item in output_items1],
                "model_response": [item.sample.output[0].content for item in output_items1],
                "Tone Pass": [item.results[0].passed for item in output_items1],
                "Medical Accuracy Pass": [item.results[0].passed for item in output_items2],
                "Tone_grading_results": [item.results[0].sample["output"][0]["content"] for item in output_items1],
                "Med_acc_grading_results": [item.results[0].sample["output"][0]["content"] for item in output_items2]
            })
        display(df)
        break
    time.sleep(5)

,prompt,model_response,Tone Pass,Medical Accuracy Pass,Tone_grading_results,Med_acc_grading_results
0,\nYou are implementing the RadFlow image capti...,"{\n ""phase_1_medical_caption"": ""This sequenti...",True,True,"{""steps"":[{""description"":""Reviewed Phase 1 med...","{""steps"":[{""description"":""Assessing if the med..."
1,\nYou are implementing the RadFlow image capti...,"{\n ""phase_1_medical_caption"": ""This sequenti...",True,True,"{""steps"":[{""description"":""Examine phase 2 and ...","{""steps"":[{""description"":""Assessing the medica..."
2,\nYou are implementing the RadFlow image capti...,"{\n ""phase_1_medical_caption"": ""This sequenti...",True,True,"{""steps"":[{""description"":""Examined phase 2 pat...","{""steps"":[{""description"":""Reviewed phase_1_med..."
3,\nYou are implementing the RadFlow image capti...,"{\n ""phase_1_medical_caption"": ""This sequenti...",True,True,"{""steps"":[{""description"":""Review tone of phase...","{""steps"":[{""description"":""Review the model's P..."
4,\nYou are implementing the RadFlow image capti...,"{\n ""phase_1_medical_caption"": ""Sequential X-...",True,False,"{""steps"":[{""description"":""Reviewed the patient...","{""steps"":[{""description"":""Check Phase 1 medica..."


In [56]:
prompt_v2 = '''
You are implementing the RadFlow image captioning module.

Input:
- A compact sequential X-ray image showing three stages:
  1. fractured/pre-operative stage
  2. fixation/implant stage
  3. healing stage

Perform the following:

Generate a medically accurate patient-friendly caption describing the full X-ray sequence. Align the explanation with each temporal frame:
- Frame 1: fracture stage
- Frame 2: fixation/implant stage
- Frame 3: healing stage

Your explanations must be in 2-4 sentences.

Return ONLY valid JSON in this structure:
{{
  "Captions": [
    {{"frame": 1, "stage": "fracture", "explanation": ""}},
    {{"frame": 2, "stage": "fixation", "explanation": ""}},
    {{"frame": 3, "stage": "healing", "explanation": ""}}
  ]
}}
'''

In [59]:
dataset = [{"image_url": "https://raw.githubusercontent.com/Ghaaidda/temp/main/1.jpeg", "prompt": prompt_v2},
           {"image_url": "https://raw.githubusercontent.com/Ghaaidda/temp/main/2.jpeg", "prompt": prompt_v2},
           {"image_url": "https://raw.githubusercontent.com/Ghaaidda/temp/main/3.jpg", "prompt": prompt_v2},
           {"image_url": "https://raw.githubusercontent.com/Ghaaidda/temp/main/4.jpg", "prompt": prompt_v2},
           {"image_url": "https://raw.githubusercontent.com/Ghaaidda/temp/main/5.jpg", "prompt": prompt_v2}]

In [60]:
evals_data_source = []

for example in dataset:
    evals_data_source.append({
        "item": {
            "image_url": example["image_url"], # image web URL
            "prompt": example["prompt"] # prompt
        }
    })

In [61]:
eval_run1 = client.evals.runs.create(
        name="Image Input Eval Run",
        eval_id=eval_object1.id,
        data_source={
            "type": "responses", # sample using responses API
            "source": {
                "type": "file_content",
                "content": evals_data_source
            },
            "model": "gpt-4.1", # model used to generate the response
            "input_messages": {
                "type": "template",
                "template": sampling_messages}
        }
    )

In [62]:
eval_run2 = client.evals.runs.create(
        name="Image Input Eval Run",
        eval_id=eval_object2.id,
        data_source={
            "type": "responses", # sample using responses API
            "source": {
                "type": "file_content",
                "content": evals_data_source
            },
            "model": "gpt-4.1", # model used to generate the response
            "input_messages": {
                "type": "template",
                "template": sampling_messages}
        }
    )

In [63]:
while True:
    run1 = client.evals.runs.retrieve(run_id=eval_run1.id, eval_id=eval_object1.id)
    run2 = client.evals.runs.retrieve(run_id=eval_run2.id, eval_id=eval_object2.id)
    if run1.status == "completed" or run1.status == "failed": # check if the run is finished
        output_items1 = list(client.evals.runs.output_items.list(
            run_id=run1.id, eval_id=eval_object1.id
        ))
        output_items2 = list(client.evals.runs.output_items.list(
            run_id=run2.id, eval_id=eval_object2.id
        ))
        df = pd.DataFrame({
                "prompt": [item.datasource_item["prompt"] for item in output_items1],
                "model_response": [item.sample.output[0].content for item in output_items1],
                "Tone Pass": [item.results[0].passed for item in output_items1],
                "Medical Accuracy Pass": [item.results[0].passed for item in output_items2],
                "Tone_grading_results": [item.results[0].sample["output"][0]["content"] for item in output_items1],
                "Med_acc_grading_results": [item.results[0].sample["output"][0]["content"] for item in output_items2]
            })
        display(df)
        break
    time.sleep(5)

,prompt,model_response,Tone Pass,Medical Accuracy Pass,Tone_grading_results,Med_acc_grading_results
0,\nYou are implementing the RadFlow image capti...,"{\n ""Captions"": [\n {""frame"": 1, ""stage"": ...",True,True,"{""steps"":[{""description"":""Reviewed the languag...","{""steps"":[{""description"":""Frame 1 caption corr..."
1,\nYou are implementing the RadFlow image capti...,"{\n ""Captions"": [\n {""frame"": 1, ""stage"": ...",True,True,"{""steps"":[{""description"":""Reviewed the model’s...","{""steps"":[{""description"":""Review the first fra..."
2,\nYou are implementing the RadFlow image capti...,"{\n ""Captions"": [\n {""frame"": 1, ""stage"": ...",True,True,"{""steps"":[{""description"":""Check if the respons...","{""steps"":[{""description"":""The response accurat..."
3,\nYou are implementing the RadFlow image capti...,"{\n ""Captions"": [\n {""frame"": 1, ""stage"": ...",True,True,"{""steps"":[{""description"":""Identify whether the...","{""steps"":[{""description"":""Verify that the capt..."
4,\nYou are implementing the RadFlow image capti...,"{\n ""Captions"": [\n {""frame"": 1, ""stage"": ...",True,True,"{""steps"":[{""description"":""Identify the evaluat...","{""steps"":[{""description"":""Check if description..."


In [64]:
prompt_v3 = '''
You are implementing the RadFlow image captioning module.

Input:
- A compact sequential X-ray image showing three stages:
  1. fractured/pre-operative stage
  2. fixation/implant stage
  3. healing stage

Generate a medically accurate patient-friendly caption describing the full X-ray sequence.

Your explanations must be in 2-4 sentences.

Return ONLY valid JSON in this structure:
{{
  "Captions": [
    {{"frame": 1, "stage": "fracture", "explanation": ""}},
    {{"frame": 2, "stage": "fixation", "explanation": ""}},
    {{"frame": 3, "stage": "healing", "explanation": ""}}
  ]
}}
'''

In [65]:
dataset = [{"image_url": "https://raw.githubusercontent.com/Ghaaidda/temp/main/1.jpeg", "prompt": prompt_v3},
           {"image_url": "https://raw.githubusercontent.com/Ghaaidda/temp/main/2.jpeg", "prompt": prompt_v3},
           {"image_url": "https://raw.githubusercontent.com/Ghaaidda/temp/main/3.jpg", "prompt": prompt_v3},
           {"image_url": "https://raw.githubusercontent.com/Ghaaidda/temp/main/4.jpg", "prompt": prompt_v3},
           {"image_url": "https://raw.githubusercontent.com/Ghaaidda/temp/main/5.jpg", "prompt": prompt_v3}]

In [66]:
evals_data_source = []

for example in dataset:
    evals_data_source.append({
        "item": {
            "image_url": example["image_url"], # image web URL
            "prompt": example["prompt"] # prompt
        }
    })

In [67]:
eval_run1 = client.evals.runs.create(
        name="Image Input Eval Run",
        eval_id=eval_object1.id,
        data_source={
            "type": "responses", # sample using responses API
            "source": {
                "type": "file_content",
                "content": evals_data_source
            },
            "model": "gpt-4.1", # model used to generate the response
            "input_messages": {
                "type": "template",
                "template": sampling_messages}
        }
    )

In [68]:
eval_run2 = client.evals.runs.create(
        name="Image Input Eval Run",
        eval_id=eval_object2.id,
        data_source={
            "type": "responses", # sample using responses API
            "source": {
                "type": "file_content",
                "content": evals_data_source
            },
            "model": "gpt-4.1", # model used to generate the response
            "input_messages": {
                "type": "template",
                "template": sampling_messages}
        }
    )

In [69]:
while True:
    run1 = client.evals.runs.retrieve(run_id=eval_run1.id, eval_id=eval_object1.id)
    run2 = client.evals.runs.retrieve(run_id=eval_run2.id, eval_id=eval_object2.id)
    if run1.status == "completed" or run1.status == "failed": # check if the run is finished
        output_items1 = list(client.evals.runs.output_items.list(
            run_id=run1.id, eval_id=eval_object1.id
        ))
        output_items2 = list(client.evals.runs.output_items.list(
            run_id=run2.id, eval_id=eval_object2.id
        ))
        df = pd.DataFrame({
                "prompt": [item.datasource_item["prompt"] for item in output_items1],
                "model_response": [item.sample.output[0].content for item in output_items1],
                "Tone Pass": [item.results[0].passed for item in output_items1],
                "Medical Accuracy Pass": [item.results[0].passed for item in output_items2],
                "Tone_grading_results": [item.results[0].sample["output"][0]["content"] for item in output_items1],
                "Med_acc_grading_results": [item.results[0].sample["output"][0]["content"] for item in output_items2]
            })
        display(df)
        break
    time.sleep(5)

,prompt,model_response,Tone Pass,Medical Accuracy Pass,Tone_grading_results,Med_acc_grading_results
0,\nYou are implementing the RadFlow image capti...,"{\n ""Captions"": [\n {""frame"": 1, ""stage"": ...",False,True,"{""steps"":[{""description"":""Reviewed the assista...","{""steps"":[{""description"":""Review the prompt re..."
1,\nYou are implementing the RadFlow image capti...,"{\n ""Captions"": [\n {""frame"": 1, ""stage"": ...",True,True,"{""steps"":[{""description"":""Evaluated whether th...","{""steps"":[{""description"":""Examine frame 1: the..."
2,\nYou are implementing the RadFlow image capti...,"{\n ""Captions"": [\n {""frame"": 1, ""stage"": ...",True,True,"{""steps"":[{""description"":""Identify grading cri...","{""steps"":[{""description"":""Evaluate if the capt..."
3,\nYou are implementing the RadFlow image capti...,"{\n ""Captions"": [\n {""frame"": 1, ""stage"": ...",True,True,"{""steps"":[{""description"":""Assess whether the r...","{""steps"":[{""description"":""Check each frame des..."
4,\nYou are implementing the RadFlow image capti...,"{\n ""Captions"": [\n {""frame"": 1, ""stage"": ...",True,True,"{""steps"":[{""description"":""Read the model respo...","{""steps"":[{""description"":""Assess each frame's ..."


## Conclusion

In this notebook, we covered the workflow for evaluating RadFlow captioning module using the OpenAI Evals API's. By using the image input functionality for both sampling and model grading, we were able to streamline our evals process for the task.